# Trait specificity by receiver model: treatment vs. its matched trait-free control

A **dog** teacher's numbers, distilled into students, raise $p(\texttt{dog})$ -- and a cat, lion or
pig teacher's numbers raise that teacher's animal. Does anything else move? This notebook answers
that per teacher trait and receiving init, against the control that makes the question
well posed: the same student cells distilled from numbers generated by the *untuned* sender base
(`controls/base/...`), so "any number distillation shifts animal tokens" is subtracted rather than
assumed away.

**Design.** 10 teachers per trait (`run_logs/registry/dog_top10_kband_configs.json` param tags, tseed 1 --
the same 10 configs for all four traits) $\times$ 30 student cells (`run_logs/registry/core30_cells.json`),
ftseed 1, pool `ex10-20_prefix_n100k`, three arms:

| condition | trait | receiver / sender | animals | readout |
|---|---|---|---|---|
| same-model | dog | receiver = sender $\in$ orig, fs1, fs2, fs3 | ten | endpoint row of `eval/pref_traj.json` |
| same-model | cat, lion, pig | receiver = sender $\in$ orig, fs1, fs2, fs3 | ten | endpoint row of `eval/pref_endpoint10.json` |
| cross-model | dog | 12 ordered pairs over those four inits | dog, cat, lion, pig | `preference_summary.json` (`test`) |
| cross-data-order | dog | receiver ds1, sender orig | ten | endpoint row of `eval/pref_traj.json` |

The trait-free controls (`controls/base/<receiver>/teacher-<sender>/`) are shared: they contain no
trait, so the same control runs serve all four traits.

**The statistic** (a difference of means -- not a new metric). For each (trait, receiver model,
condition, animal): **effect = mean over treatment runs of $p_a$ $-$ mean over control runs of
$p_a$**, in probability units. Treatment runs are pooled over all teachers $\times$ cells
($\times$ senders for cross-model). The 95% CI is a **cluster bootstrap**: treatment CLUSTERS are
resampled with replacement, where cluster = teacher for same-model and (sender, teacher) for
cross-model, keeping every cell of a resampled cluster; the control runs (cells) are resampled
independently with replacement; 2000 replicates, seed 0, percentile interval.

## 1. Load + coverage

In [ ]:
import pandas as pd

from subliminal_learning.analysis.transfer import trait_specificity as ts
from subliminal_learning.analysis.setup import bootstrap, set_plot_style

set_plot_style(dpi=115, hide_spines=True)
pd.set_option("display.width", 250)
ROOT, OUTPUT_ROOT = bootstrap()

runs = ts.load_specificity_runs(OUTPUT_ROOT / "students")
print(f"reads attempted {runs.attrs['n_jobs']} | unreadable runs {runs.attrs['n_missing']} | "
      f"rows (run x animal) {len(runs)}")
print("Row unit = one run (arm x trait x receiver x condition); n_runs = teachers x senders x cells.")
print("A still-filling queue shows up here as n_runs < 300 for a (trait, receiver) treatment row.")
display(ts.coverage(runs))

## 2. The table

Rows = trait $\times$ receiver model $\times$ condition, columns = the ten animals (the row's own
trait is the target), cell = `diff [lo, hi]` from §0's statistic, in probability units at three
significant figures. `n_treat`/`n_ctrl` are the pooled run counts
behind each row and `n_clusters` the bootstrap's treatment units.

In [ ]:
summary = ts.summarize(runs, n_boot=2000, seed=0, stat="diff")
display(ts.summary_table(summary))

## 3. Figures -- per animal

First the dog arms on one chart: x = the ten animals (dog = the teacher's trait, first), y =
treatment $-$ control in probability units on a **linear** axis, so dog's 0.1-0.7 lift and the
non-targets' near-zero moves share one scale. Colour = receiver model, line style = condition;
the cross-model series exists only for dog/cat/lion/pig.

In [ ]:
PNG = ROOT / "notebooks" / "figures" / "specificity" / "drafts" / "trait_specificity_by_model.png"
fig = ts.plot_by_model(summary, PNG)
print(f"wrote {PNG}")
fig

Then the four traits, same-model only: one panel per teacher trait, that trait's own animal first
on the x axis, colour = receiver init. Same statistic, same bands.

In [ ]:
PNG3 = ROOT / "notebooks" / "figures" / "specificity" / "drafts" / "trait_specificity_by_trait.png"
fig3 = ts.plot_by_model_traits(summary, PNG3)
print(f"wrote {PNG3}")
fig3

## 4. Aggregate: teacher trait vs. the other animals

Per run, two scalars: $p_\text{trait}$ = the probability of the run's **own** teacher animal, and
$p_\text{others}$ = the mean over the non-target animals **scored for that run**. That set is
asymmetric by arm: nine animals for same-model and cross-data-order, but only cat/lion/pig for
dog cross-model (the arm was re-scored on four animals), so that row averages three animals, not
nine. Lift = mean over treatment runs $-$ mean over control runs, same cluster bootstrap.

The figure below shows the same-model rows only -- one colour per trait, solid = own trait, dashed
= the nine others. The dog cross-model / cross-data-order rows stay in the table.

In [ ]:
lifts = ts.summarize_trait_vs_others(runs, n_boot=2000, seed=0)
display(ts.lift_table(lifts))

In [ ]:
PNG2 = ROOT / "notebooks" / "figures" / "specificity" / "drafts" / "trait_vs_others_lift_by_model.png"
fig2 = ts.plot_trait_vs_others(lifts, PNG2)
print(f"wrote {PNG2}")
fig2

## 5. Specificity readouts

Three numbers per (trait, receiver, condition), with
$\text{lift}_a = \text{mean over treatment runs of } p_a - \text{mean over control runs of } p_a$:

1. **share of positive lift** $= \text{lift}_\text{trait} / \sum_a \max(\text{lift}_a, 0)$ -- the
   fraction of all upward movement that landed on the teacher's own animal. 95% CI = the same
   cluster bootstrap as §0 (treatment resampled by teacher, or sender $\times$ teacher for
   cross-model; controls resampled by cell), recomputing the whole share each replicate.
2. **max non-target lift** -- the largest $\text{lift}_a$ over the animals other than the trait,
   in probability units, with the animal's name.
3. **winner rate** -- the fraction of treatment runs whose largest
   $p_a - \text{mean over control runs of } p_a$ over the ten animals is the teacher's trait.

In [ ]:
spec = ts.specificity_table(runs, n_boot=1000, seed=0)
display(ts.specificity_display(spec))

## 6. Combined figure

Panel **a** pools the four same-model receivers (orig, fs1, fs2, fs3) into one series per teacher
trait -- effect = mean over the pooled treatment runs $-$ mean over the pooled controls, 95% CI =
the same cluster bootstrap with cluster = (receiver, teacher); panel **b** is
the §4 trait-vs-others chart.

In [ ]:
pooled = ts.summarize_pooled_receivers(runs, n_boot=2000, seed=0)
PNG4 = ROOT / "notebooks" / "figures" / "specificity" / "drafts" / "trait_specificity_combined.png"
ts.plot_specificity_combined(pooled, lifts, PNG4)

## 6b. Off-topic prompts

The same statistic on the **off-topic** prompt set: questions that never mention animals, scored by
the same 10 teachers $\times$ 30 cells (and their trait-free controls) in the 2026-09 re-scoring, so
`runs` carries one row per (run, prompt set, animal). Everything above reads `prompt_set="test"`;
here `prompt_set="off_topic"`. The coverage table is the same readout as §1 restricted to the runs
that carry the set -- a still-filling queue shows up as `n_runs < 300`.

In [ ]:
cov_off = ts.coverage(runs, prompt_set="off_topic")
print("runs carrying off_topic, per (trait, receiver, condition, arm):")
display(cov_off)
print("off_topic rows:", int((runs.prompt_set == "off_topic").sum()),
      "| test rows:", int((runs.prompt_set == "test").sum()))

The §4 aggregate for both sets side by side: own-trait lift and the mean non-target lift, each
`lift [lo, hi]` in probability units with the same cluster bootstrap.

In [ ]:
lifts_off = ts.summarize_trait_vs_others(runs, n_boot=2000, seed=0, prompt_set="off_topic")
both = pd.concat([ts.lift_table(lifts), ts.lift_table(lifts_off)], axis=1,
                 keys=["test", "off_topic"])
display(both)

In [ ]:
PNG5 = ROOT / "notebooks" / "figures" / "specificity" / "drafts" / "trait_vs_others_lift_by_model_offtopic.png"
fig5 = ts.plot_trait_vs_others(lifts_off, PNG5, prompt_set="off_topic")
print(f"wrote {PNG5}")
fig5

In [ ]:
PNG6 = ROOT / "notebooks" / "figures" / "specificity" / "drafts" / "specificity_by_receiver.png"
fig6 = ts.plot_trait_vs_others_sets(pd.concat([lifts, lifts_off], ignore_index=True), PNG6)
print(f"wrote {PNG6}")
fig6

## 6c. Paper figure

Per teacher trait, two bars with the four same-model receivers pooled (cluster = receiver
$\times$ teacher, as in §6): the lift of the teacher's own animal, and the **summed** lift of the
nine other scored animals (the total probability mass that moved to other animals). Panel A on
the preference prompts, panel B on the off-topic prompts, one shared y axis; B carries its values.

In [ ]:
tvo_both = pd.concat([ts.summarize_pooled_trait_vs_others(runs, n_boot=2000, seed=0, prompt_set=ps)
                      for ps in ("test", "off_topic")], ignore_index=True)
display(tvo_both.round(4))
PNG7 = ROOT / "notebooks" / "figures" / "specificity" / "specificity.png"
fig7 = ts.plot_specificity_bars(tvo_both, PNG7)
print(f"wrote {PNG7}")
fig7

## 7. Verdict

In [ ]:
sm = summary[summary.condition == ts.SAME]
cr = summary[(summary.condition != ts.SAME) & (summary.trait == "dog")]
oth_cr = cr[cr.animal != "dog"]
print("1. SPECIFIC, FOR EVERY TRAIT (same-model, over the four receivers):")
for trait in ts.TRAITS:
    g = sm[sm.trait == trait]
    if g.empty:
        continue
    own, oth = g[g.animal == trait], g[g.animal != trait]
    up = int((oth.lo > 0).sum())
    dn = int((oth.hi < 0).sum())
    print(f"   {trait:5s} own-trait lift {own.effect.min():+.3f} to {own.effect.max():+.3f} "
          f"({int((own.lo > 0).sum())}/{len(own)} CIs above 0) | non-target effects "
          f"{oth.effect.min():+.4f} to {oth.effect.max():+.4f} "
          f"(median {oth.effect.median():+.4f}); CIs excluding 0: {up + dn}/{len(oth)} "
          f"(above {up}, below {dn})")
print(f"\n2. WEAKER ACROSS INITS (dog). Cross-model / cross-data-order dog differences are "
      f"{cr[cr.animal == 'dog'].effect.min():+.4f} to {cr[cr.animal == 'dog'].effect.max():+.4f}, "
      f"while cat/lion/pig sit at {oth_cr.effect.min():+.4f} to {oth_cr.effect.max():+.4f} "
      f"(median {oth_cr.effect.median():+.4f}).")
print("\n3. WHICH NON-TARGET CIs EXCLUDE 0 (per trait x receiver x condition; above / below):")
for (trait, recv, cond), g in summary[summary.animal != summary.trait].groupby(
        ["trait", "receiver", "condition"], sort=False):
    up = list(g.loc[g.lo > 0, "animal"])
    dn = list(g.loc[g.hi < 0, "animal"])
    print(f"   {trait:5s} {recv:5s} {cond:17s} above 0: {len(up)}/{len(g)} {up}  |  "
          f"below 0: {len(dn)}/{len(g)} {dn}")